# Bios Data Cleaning

This notebook cleans and preprocesses the raw `bios.csv` dataset.

The cleaning process focuses on athlete names, physical measurements, birth and death information, and categorical variables.

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import re

## 2. Load Raw Data

The raw athlete biography dataset is loaded from `bios.csv`.

A copy is created as `bios` so that the original raw dataset remains unchanged during the cleaning process.

In [109]:
bios_df = pd.read_csv("./bios.csv")
bios = bios_df.copy()

## 3. Initial Data Inspection

The initial structure of the dataset is inspected to understand the number of rows, columns, data types, and missing values before cleaning.

In [110]:
bios_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 145500 entries, 0 to 145499
Data columns (total 16 columns):
 #   Column         Non-Null Count   Dtype
---  ------         --------------   -----
 0   Roles          145500 non-null  str  
 1   Sex            145500 non-null  str  
 2   Full name      145500 non-null  str  
 3   Used name      145500 non-null  str  
 4   Born           143772 non-null  str  
 5   Died           34042 non-null   str  
 6   NOC            145499 non-null  str  
 7   athlete_id     145500 non-null  int64
 8   Measurements   107833 non-null  str  
 9   Affiliations   95832 non-null   str  
 10  Nick/petnames  9145 non-null    str  
 11  Title(s)       399 non-null     str  
 12  Other names    7167 non-null    str  
 13  Nationality    8259 non-null    str  
 14  Original name  30739 non-null   str  
 15  Name order     7844 non-null    str  
dtypes: int64(1), str(15)
memory usage: 17.8 MB


## 4. Data Cleaning

### 4.1 Name Cleaning

The `Used name` and `Full name` columns contain inconsistent whitespace and bullet characters.

Bullet characters are replaced with spaces, consecutive whitespace is normalized, and leading/trailing whitespace is removed.

In [111]:
bios['Used name_clean'] = (
        bios['Used name']
        .str.replace('•', ' ')
        .str.replace(r'\s+', ' ')
        .str.strip()
)

bios['Full name_clean'] = (
    bios['Full name']
    .str.replace('•', ' ')
    .str.replace(r'\s+', ' ')
    .str.strip()
)

### 4.2 Measurements Cleaning

The `Measurements` column contains height and weight information in different text formats.

Height values are extracted in centimeters, while weight values are converted into numeric minimum and maximum values in kilograms.

Weight ranges are normalized so that the smaller value is stored in `weight_min_kg` and the larger value in `weight_max_kg`.

In [112]:
height = bios['Measurements'].str.extract(
    r'(\d+(?:[.,]\d+)?)\s*cm',
    expand=False
)

bios['Height_cm'] = (
    height
    .str.replace(',', '.', regex=False)
    .astype(float)
)

weight_text = bios['Measurements'].str.extract(
    r'(?:/\s*)?([\d\s,–-]+)\s*kg',
    expand=False
)

In [2]:
# Parse single weights, decimal values, and weight ranges.
def parse_weight(value):
    if pd.isna(value):
        return np.nan, np.nan

    value = value.strip()

    if re.fullmatch(r'\d+,\d+', value):
        weight = float(value.replace(',', '.'))
        return weight, weight

    if re.fullmatch(r'\d+\s*[–-]\s*\d+', value):
        numbers = [float(x) for x in re.findall(r'\d+', value)]
        return min(numbers), max(numbers)

    numbers = re.findall(r'\d+', value)

    if numbers:
        numbers = [float(x) for x in numbers]
        return min(numbers), max(numbers)

    return np.nan, np.nan

In [114]:
weights = weight_text.apply(parse_weight)

bios['Weight_min_kg'] = weights.str[0]
bios['Weight_max_kg'] = weights.str[1]

### 4.3 Birth Information

The `Born` column contains birth dates, years, places, and country codes in text format.

The information is separated into four analysis-ready columns:

- `Birth_date`
- `Birth_year`
- `Birth_place`
- `Birth_country`

Full dates are converted to datetime values, while records without a complete date retain the available birth year when it can be extracted.

In [115]:
birth_date_text = bios['Born'].str.extract(
    r'(\d{1,2}\s+[A-Za-z]+\s+\d{4})',
    expand=False
)

bios['Birth_date'] = pd.to_datetime(
    birth_date_text,
    format='%d %B %Y',
    errors='coerce'
)

bios['Birth_year'] = (
    bios['Born']
    .str.extract(r'(\d{4})', expand=False)
    .astype('Int64')
)

bios['Birth_place'] = bios['Born'].str.extract(
    r'(?:^|\s)in\s+(.+?)\s*,?\s*\([A-Z]{3}\)$',
    expand=False
)

bios['Birth_country'] = bios['Born'].str.extract(
    r'\(([A-Z]{3})\)$',
    expand=False
)

### 4.4 Death Information

The `Died` column contains different formats of death information, including complete dates, years, and uncertain descriptions.

Complete dates are converted to datetime values. When a complete date is unavailable, a reliable year is extracted where possible.

Records containing uncertain descriptions in parentheses are not assigned a death year.

In [118]:
died_date_text = bios['Died'].str.extract(
    r'^(\d{1,2}\s+[A-Za-z]+\s+\d{4})',
    expand=False
)

bios['Died_date'] = pd.to_datetime(
    died_date_text,
    format='%d %B %Y',
    errors='coerce'
)

unclassified = bios['Died'].str.startswith('(', na=False)

bios['Died_year'] = (
    bios['Died']
    .where(~unclassified)
    .str.extract(r'(\d{4})', expand=False)
    .astype('Int64')
)

### 4.5 Sex

The `Sex` column is standardized using short categorical codes:

- `Male` → `M`
- `Female` → `F`

In [119]:
bios['Sex'] = bios['Sex'].map({
    'Male': 'M',
    'Female': 'F'
})

### 4.6 Remove Unnecessary Columns

Columns that are not required for the analysis are removed.

The original text columns used during the cleaning process are also removed after their information has been extracted into structured columns.

The raw `bios.csv` file remains unchanged, so the original information is preserved.

In [120]:
bios = bios.drop(columns=[
    'Name order',
    'Title(s)',
    'Nick/petnames',
    'Other names',
    'Nationality',
    'Original name',
    'Full name',
    'Used name',
    'Born',
    'Died',
    'Measurements'
])

### 4.7 Standardize Column Names

The cleaned columns are renamed and all column names are converted to lowercase `snake_case` format.

This provides a consistent naming convention for the analysis and improves compatibility with Python and pandas workflows.

In [121]:
bios = bios.rename(columns={
    'Full name_clean': 'full_name',
    'Used name_clean': 'used_name',
    'Birth_date': 'birth_date',
    'Birth_year': 'birth_year',
    'Birth_place': 'birth_place',
    'Birth_country': 'birth_country',
    'Died_date': 'died_date',
    'Died_year': 'died_year',
    'Height_cm': 'height_cm',
    'Weight_min_kg': 'weight_min_kg',
    'Weight_max_kg': 'weight_max_kg'
})

In [122]:
bios.columns = (
    bios.columns
    .str.strip()
    .str.lower()
    .str.replace(r'[\s/()-]+', '_', regex=True)
    .str.replace(r'_+', '_', regex=True)
    .str.strip('_')
)

## 5. Final Data Check

The cleaned dataset is inspected to verify its final structure, data types, sample records, and missing values before export.

In [123]:
bios.info()

<class 'pandas.DataFrame'>
RangeIndex: 145500 entries, 0 to 145499
Data columns (total 16 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   roles          145500 non-null  str           
 1   sex            145500 non-null  str           
 2   noc            145499 non-null  str           
 3   athlete_id     145500 non-null  int64         
 4   affiliations   95832 non-null   str           
 5   used_name      145500 non-null  str           
 6   full_name      145500 non-null  str           
 7   height_cm      106651 non-null  float64       
 8   weight_min_kg  104210 non-null  float64       
 9   weight_max_kg  104210 non-null  float64       
 10  birth_date     141685 non-null  datetime64[us]
 11  birth_year     143698 non-null  Int64         
 12  birth_place    120926 non-null  str           
 13  birth_country  120926 non-null  str           
 14  died_date      31925 non-null   datetime64[us]
 15  died_year  

In [128]:
bios.head(5)

,roles,sex,noc,athlete_id,affiliations,used_name,full_name,height_cm,weight_min_kg,weight_max_kg,birth_date,birth_year,birth_place,birth_country,died_date,died_year
0,Competed in Olympic Games,M,France,1,NaN,Jean-François Blanchy,"François Joseph Marie Antoine ""Jean-François"" ...",NaN,NaN,NaN,1886-12-12,1886,"Bordeaux, Gironde",FRA,1960-10-02,1960
1,Competed in Olympic Games,M,France,2,"Racing Club de France, Paris (FRA)",Arnaud Boetsch,Arnaud Benjamin Boetsch,183.0,76.0,76.0,1969-04-01,1969,"Meulan, Yvelines",FRA,NaT,<NA>
2,Competed in Olympic Games • Administrator,M,France,3,"TCP, Paris (FRA)",Jean Borotra,Jean Laurent Robert Borotra,183.0,76.0,76.0,1898-08-13,1898,"Biarritz, Pyrénées-Atlantiques",FRA,1994-07-17,1994
3,Competed in Olympic Games,M,France,4,"Sporting club de Paris, Paris (FRA)",Jacques Brugnon,Jacques Marie Stanislas Jean Brugnon,168.0,64.0,64.0,1895-05-11,1895,"Paris VIIIe, Paris",FRA,1978-03-20,1978
4,Competed in Olympic Games,M,France,5,"TCP, Paris (FRA)",Albert Canet,Henry Albert Canet,NaN,NaN,NaN,1878-04-17,1878,"Wandsworth, England",GBR,1930-07-25,1930


In [130]:
bios.isna().sum()

roles                 0
sex                   0
noc                   1
athlete_id            0
affiliations      49668
used_name             0
full_name             0
height_cm         38849
weight_min_kg     41290
weight_max_kg     41290
birth_date         3815
birth_year         1802
birth_place       24574
birth_country     24574
died_date        113575
died_year        111702
dtype: int64

## 6. Export Cleaned Data

The cleaned dataset is exported as `bios_clean.csv`.

The original `bios.csv` file is preserved as the raw dataset.

In [129]:
bios.to_csv("./bios_clean.csv", index=False)